# PCA as a featurizer: squashing the data pancake

_Feature Engineering_

**Find the few directions where the data spreads out the most, keep those, and drop the rest.**

Imagine your data as a cloud of points in many dimensions. Often that cloud is shaped like a
       pancake: it is wide in a couple of directions and almost flat in all the others. If a
       direction is flat, the points barely differ along it &mdash; so that direction carries almost no
       information. You could throw it away and lose almost nothing.

       PCA (Principal Component Analysis) is the recipe for finding those directions. It is
       Chapter 6 of Zheng & Casari's Feature Engineering for Machine Learning, and in that book
       PCA is treated as a featurizer: a transform that takes your many raw columns and returns a
       handful of new ones. The new columns are called principal components. They are new axes,
       ordered by how much the data spreads out (its variance) along each. Keep the top few and you
       have squashed a high-dimensional pancake onto a low-dimensional plane &mdash; smaller, decorrelated,
       and still carrying most of the signal.

---

This notebook is a practice scaffold for the **PCA as a featurizer: squashing the data pancake** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# fe-pca.py
# LESSON: PCA (Principal Component Analysis) — reduce many features to a few,
#         as a featurizer for a downstream model.
#
# PROBLEM we reproduce: the digits dataset has 64 pixel features. Many pixels are
# redundant (neighbours look alike) or near-constant (corners are always blank).
# Feeding all 64 to a classifier is wasteful: it trains SLOWER and high
# dimensionality can hurt when data is limited. We also show the data is highly
# COMPRESSIBLE — a cumulative explained-variance curve shows a handful of PCA
# components already capture ~90% of the variance.
# FIX: PCA down to the few components that reach ~90% variance, then refit the
# SAME classifier. We get comparable accuracy with FAR fewer features (faster).
# KEY: we fit the scaler AND PCA on the TRAIN split ONLY, then apply them to the
# test split. Fitting on all the data would leak test info — a classic mistake.
#
# Runs top-to-bottom in Colab. Packages: numpy, matplotlib, scikit-learn only.

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------------------
# STEP 1: Load real data. 1797 handwritten digits, each an 8x8 image flattened
# to 64 pixel features (values 0..16). y is the digit (0..9).
# ---------------------------------------------------------------------------
data = load_digits()
X, y = data.data, data.target               # X: 1797 rows x 64 pixel features
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

# Standardize on TRAIN ONLY (fit on train, then apply to both). No leakage.
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

# ---------------------------------------------------------------------------
# STEP 2: Visualize the PURE (raw) data — show a few raw 8x8 digit images so the
# beginner sees what one "row of 64 features" actually is.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, img, label in zip(axes, data.images[:6], y[:6]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"digit {label}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("STEP 2: raw digits — each is 64 pixel features")
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# STEP 3: Reproduce the PROBLEM — fit the classifier on ALL 64 raw features.
# Record accuracy AND training time. 64 dims is wasteful and slower.
# ---------------------------------------------------------------------------
raw_clf = LogisticRegression(max_iter=5000, random_state=0)
t0 = time.perf_counter()
raw_clf.fit(X_tr_s, y_tr)
raw_time = time.perf_counter() - t0
raw_acc = accuracy_score(y_te, raw_clf.predict(X_te_s))
raw_dims = X_tr_s.shape[1]                   # = 64

# ---------------------------------------------------------------------------
# STEP 4: Apply PCA (fit on TRAIN ONLY) and visualize.
# (a) Cumulative explained-variance curve: how much total variance the first k
#     components capture. Pick the smallest k reaching 90% -> our cutoff.
# (b) A 2-component PCA scatter: project to 2D and colour by digit class, to SEE
#     that PCA keeps class structure even after a huge dimensionality drop.
# ---------------------------------------------------------------------------
pca_full = PCA(random_state=0).fit(X_tr_s)   # fit on TRAIN only
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.searchsorted(cum_var, 0.90) + 1)  # smallest #components reaching 90%

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, len(cum_var) + 1), cum_var, "o-", ms=3, color="steelblue")
ax[0].axhline(0.90, color="gray", ls=":", lw=1)
ax[0].axvline(k, color="red", ls="--", lw=1, label=f"cutoff = {k} comps (~90%)")
ax[0].set_title("STEP 4a: cumulative explained variance vs #components")
ax[0].set_xlabel("number of PCA components"); ax[0].set_ylabel("cumulative variance")
ax[0].legend()

X_tr_2d = PCA(n_components=2, random_state=0).fit(X_tr_s).transform(X_tr_s)
sc = ax[1].scatter(X_tr_2d[:, 0], X_tr_2d[:, 1], c=y_tr, cmap="tab10", s=8, alpha=0.7)
ax[1].set_title("STEP 4b: first 2 PCA components, coloured by digit")
ax[1].set_xlabel("PC 1"); ax[1].set_ylabel("PC 2")
fig.colorbar(sc, ax=ax[1], label="digit class")
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# STEP 5: Show the FIX — PCA to k components (fit on TRAIN ONLY), refit the SAME
# classifier. Comparable accuracy, far fewer features, faster training.
# ---------------------------------------------------------------------------
pca = PCA(n_components=k, random_state=0).fit(X_tr_s)   # fit on TRAIN only
X_tr_p = pca.transform(X_tr_s)
X_te_p = pca.transform(X_te_s)

fix_clf = LogisticRegression(max_iter=5000, random_state=0)
t0 = time.perf_counter()
fix_clf.fit(X_tr_p, y_tr)
fix_time = time.perf_counter() - t0
fix_acc = accuracy_score(y_te, fix_clf.predict(X_te_p))

print(f"PROBLEM (raw):  dims={raw_dims}  acc={raw_acc:.3f}  train_time={raw_time:.3f}s")
print(f"FIX (PCA {k}):   dims={k}  acc={fix_acc:.3f}  train_time={fix_time:.3f}s")
print(f"PROBLEM (raw): {raw_acc:.3f}   →   FIX (engineered): {fix_acc:.3f}")

## Reference implementation — scikit-learn + numpy

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# --- MNIST-like handwritten digits: 8x8 images flattened to 64 pixel features ---
# (Chapter 6 uses image data; load_digits ships with scikit-learn.)
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# === Step 1: STANDARDIZE first (PCA is scale-sensitive) ===
# Fit the scaler on TRAIN ONLY, then reuse it -- no leakage.
scaler = StandardScaler().fit(X_train)
X_train_std = scaler.transform(X_train)
X_test_std = scaler.transform(X_test)

# === Step 2: choose how many components from the explained-variance curve ===
pca_full = PCA().fit(X_train_std)
cum = np.cumsum(pca_full.explained_variance_ratio_)
k95 = int(np.argmax(cum >= 0.95)) + 1
print('components to reach 95%% variance:', k95)   # ~28 on the digits
print('PC1 / PC2 ratio:', pca_full.explained_variance_ratio_[:2])

# === Step 3: PCA as a featurizer -- fit_transform to the reduced matrix ===
pca = PCA(n_components=k95)
X_train_pca = pca.fit_transform(X_train_std)        # n_train x k feature matrix
X_test_pca = pca.transform(X_test_std)              # apply the SAME components
print('reduced shape:', X_train_pca.shape)          # (1347, ~28) from 64

# === Whitening: decorrelated, unit-variance components ===
pca_white = PCA(n_components=k95, whiten=True)
X_train_white = pca_white.fit_transform(X_train_std)

# === Project to 2-D just to VISUALIZE the digit classes ===
X_2d = PCA(n_components=2).fit_transform(X_train_std)
# scatter X_2d colored by y_train -> the 0..9 classes pull apart

## Visualize the data & results

_On load_digits (64-dim images), how many principal components do we need before the cumulative explained variance reaches 95% -- i.e. how far can we squash the pancake while keeping the signal?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = load_digits().data                               # 1797 x 64 pixel features
X_std = StandardScaler().fit_transform(X)            # standardize: PCA is scale-sensitive

pca = PCA().fit(X_std)                               # all 64 components
cum = np.cumsum(pca.explained_variance_ratio_)       # cumulative variance kept

k95 = int(np.argmax(cum >= 0.95)) + 1
print('components for 95%% variance:', k95)          # -> ~28
for k in [1, 2, 3, 5, 10, 20, 28, 40]:
    print(f'k={k:>2}  cumulative variance = {cum[k-1]:.3f}')
# k= 1  cumulative variance = 0.149
# k= 2  cumulative variance = 0.285
# k= 3  cumulative variance = 0.403
# k= 5  cumulative variance = 0.545
# k=10  cumulative variance = 0.738
# k=20  cumulative variance = 0.894
# k=28  cumulative variance = 0.950
# k=40  cumulative variance = 0.988

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You run PCA on a table with age (years, 18&ndash;90) and income (dollars, up to 200,000) without scaling. PC1 turns out to be essentially just income. Why, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that PCA picks directions that maximize variance, measured in the raw units. — _Income in dollars has variance in the billions; age in years has variance in the hundreds._
- See that income's huge numeric spread dominates, so PC1 aligns with income regardless of which feature carries more signal. — _PCA is comparing raw spreads, not importance &mdash; it is fooled by units._
- Standardize every column to zero mean and unit variance before PCA (StandardScaler). — _Now each feature contributes spread on the same scale, so PCA ranks directions by real structure, not by unit size._

**Answer:** PCA maximizes variance in the raw units, and income's dollar-scale variance is astronomically larger than age's, so PC1 collapses onto income. The fix is to standardize (zero mean, unit variance) every feature first; PCA is scale-sensitive, so standardization is mandatory before applying it.

</details>

**Problem 2.** On load_digits (64 pixel features), the cumulative explained-variance curve reaches 0.95 at about 28 components. How do you use this to choose $k$, and what did you gain?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the cumulative curve: it sums explained_variance_ratio_ as you add components. — _The value at $k$ is the fraction of total variance the top $k$ components keep._
- Pick the smallest $k$ whose cumulative variance meets your threshold &mdash; here $k\approx 28$ for 95%. — _That is the elbow of "enough signal kept" against "as few features as possible"._
- Replace the 64 raw pixels with these 28 components as the model's input. — _Fewer than half the features, yet 95% of the spread retained &mdash; faster training, less overfitting, decorrelated inputs._

**Answer:** Plot the cumulative explained-variance curve and take the smallest $k$ that clears your threshold &mdash; about 28 components for 95% on the digits. You compress 64 correlated pixels to 28 orthogonal features, keeping 95% of the variance while cutting the dimension by more than half: faster, leaner, decorrelated input for the downstream model.

</details>

**Problem 3.** A classifier on the raw features beats the same classifier on PCA-reduced features, even though PCA kept 99% of the variance. What unsupervised blind spot of PCA could explain this?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall PCA is unsupervised &mdash; it ranks directions by variance, never looking at the labels. — _The directions it keeps are the ones with the most spread, not the ones that separate the classes._
- Notice the class-separating signal might lie in a low-variance direction that PCA discarded. — _That 1% of variance PCA threw away could be exactly the direction that tells the classes apart._
- Switch to a supervised reducer (LDA) or skip reduction, and validate accuracy against the raw-feature baseline. — _LDA chooses directions that maximize class separation; validation confirms the reduction did not drop the useful signal._

**Answer:** PCA is unsupervised: it keeps high-variance directions and ignores the labels. The class-discriminative signal can sit in a low-variance direction that PCA discarded, so "99% of variance kept" does not guarantee "99% of the useful-for-classification signal kept". Use a supervised reducer like LDA, or validate the reduction against the raw-feature baseline before trusting it.

</details>